# Tải và làm sạch bốn chuỗi giá

Notebook này tải VN-Index, BTC-USD và USD/VND; giá vàng được đọc từ file CSV do người dùng cung cấp. Notebook không gọi crawler PNJ, không ghép các tài sản và không tính return.

Bốn file đầu ra dùng trực tiếp cho notebook `xử_lý_lại_data_sau_khi_tải.ipynb`:

- `3y-bitcoin-price-clean.csv`
- `3y-usdvnd-price-clean.csv`
- `3y-sjc-price-clean.csv`
- `3y-vnindex-data-clean.csv`


## 1. Cài thư viện


In [ ]:
%pip install -q yfinance pandas numpy
%pip install -q git+https://github.com/thinh-vu/vnstock


## 2. Cấu hình chung


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from vnstock import Quote


START_DATE = "2023-08-01"
END_DATE = "2026-08-01"  # Ngày cuối cùng cần giữ, tính cả ngày này.
DOWNLOAD_END_DATE = (
    pd.Timestamp(END_DATE) + pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

BTC_OUTPUT = Path("3y-bitcoin-price-clean.csv")
USDVND_OUTPUT = Path("3y-usdvnd-price-clean.csv")
GOLD_OUTPUT = Path("3y-sjc-price-clean.csv")
VNINDEX_OUTPUT = Path("3y-vnindex-data-clean.csv")

# Chấp nhận file vàng ở data/raw/, cạnh notebook hoặc thư mục cha.
GOLD_INPUT_CANDIDATES = [
    Path("data/raw/3y-sjc.csv"),
    Path("3y-sjc.csv"),
    Path("../3y-sjc.csv"),
]


def flatten_yahoo_columns(data):
    data = data.reset_index()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    return data.loc[:, ~data.columns.duplicated()].copy()


def normalize_dates(values):
    return (
        pd.to_datetime(values, errors="coerce", utc=True)
        .dt.tz_localize(None)
        .dt.normalize()
    )


def keep_requested_dates(data):
    return data.loc[
        data["Date"].between(pd.Timestamp(START_DATE), pd.Timestamp(END_DATE))
    ].copy()


## 3. Tải và làm sạch BTC-USD


In [ ]:
btc = yf.download(
    tickers="BTC-USD",
    start=START_DATE,
    end=DOWNLOAD_END_DATE,
    interval="1d",
    auto_adjust=False,
    actions=False,
    progress=False,
    threads=False,
)
if btc.empty:
    raise RuntimeError("Không tải được BTC-USD từ Yahoo Finance.")

btc = flatten_yahoo_columns(btc)
required = ["Date", "Open", "High", "Low", "Close"]
missing = [column for column in required if column not in btc.columns]
if missing:
    raise ValueError(f"BTC-USD thiếu các cột bắt buộc: {missing}")

btc["Date"] = normalize_dates(btc["Date"])
numeric_columns = ["Open", "High", "Low", "Close"]
optional_numeric = [column for column in ["Adj Close", "Volume"] if column in btc]
for column in numeric_columns + optional_numeric:
    btc[column] = pd.to_numeric(btc[column], errors="coerce")

btc = btc.replace([np.inf, -np.inf], np.nan)
btc = btc.dropna(subset=["Date"] + numeric_columns).copy()
btc = btc.loc[btc[numeric_columns].gt(0).all(axis=1)].copy()
valid_ohlc = (
    (btc["High"] >= btc[["Open", "Low", "Close"]].max(axis=1))
    & (btc["Low"] <= btc[["Open", "High", "Close"]].min(axis=1))
)
btc = btc.loc[valid_ohlc].copy()
if "Volume" in btc:
    btc.loc[btc["Volume"] < 0, "Volume"] = np.nan

btc = keep_requested_dates(btc)
btc = (
    btc.sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
    .rename(columns={"Close": "btc_price"})
)
selected = ["Date", "Open", "High", "Low", "btc_price"]
selected.extend(column for column in ["Adj Close", "Volume"] if column in btc)
btc = btc[selected].copy()
btc["day_gap"] = btc["Date"].diff().dt.days
btc.to_csv(BTC_OUTPUT, index=False, date_format="%Y-%m-%d")
print(f"Đã lưu {len(btc):,} dòng vào {BTC_OUTPUT}")


## 4. Tải và làm sạch USD/VND


In [ ]:
usdvnd = yf.download(
    tickers="VND=X",
    start=START_DATE,
    end=DOWNLOAD_END_DATE,
    interval="1d",
    auto_adjust=False,
    actions=False,
    progress=False,
    threads=False,
)
if usdvnd.empty:
    raise RuntimeError("Không tải được USD/VND từ Yahoo Finance.")

usdvnd = flatten_yahoo_columns(usdvnd)
if not {"Date", "Close"}.issubset(usdvnd.columns):
    raise ValueError("USD/VND thiếu cột Date hoặc Close.")

usdvnd["Date"] = normalize_dates(usdvnd["Date"])
usdvnd["usdvnd_rate"] = pd.to_numeric(usdvnd["Close"], errors="coerce")
usdvnd = usdvnd.replace([np.inf, -np.inf], np.nan)
usdvnd = usdvnd.dropna(subset=["Date", "usdvnd_rate"])
usdvnd = usdvnd.loc[usdvnd["usdvnd_rate"] > 0, ["Date", "usdvnd_rate"]]
usdvnd = keep_requested_dates(usdvnd)
usdvnd = (
    usdvnd.sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)
usdvnd.to_csv(USDVND_OUTPUT, index=False, date_format="%Y-%m-%d")
print(f"Đã lưu {len(usdvnd):,} dòng vào {USDVND_OUTPUT}")


## 5. Tải và làm sạch VN-Index


In [ ]:
quote = Quote(symbol="VNINDEX", source="KBS")
vnindex_raw = quote.history(
    symbol="VNINDEX",
    start=START_DATE,
    end=DOWNLOAD_END_DATE,
    interval="1D",
)
if vnindex_raw is None or vnindex_raw.empty:
    raise RuntimeError("Không tải được VN-Index từ nguồn KBS.")
if not {"time", "close"}.issubset(vnindex_raw.columns):
    raise ValueError("VN-Index thiếu cột time hoặc close.")

vnindex = pd.DataFrame(
    {
        "Date": normalize_dates(vnindex_raw["time"]),
        "vnindex_price": pd.to_numeric(vnindex_raw["close"], errors="coerce"),
    }
)
vnindex = vnindex.replace([np.inf, -np.inf], np.nan)
vnindex = vnindex.dropna(subset=["Date", "vnindex_price"])
vnindex = vnindex.loc[vnindex["vnindex_price"] > 0]
vnindex = keep_requested_dates(vnindex)
vnindex = (
    vnindex.sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)
vnindex.to_csv(VNINDEX_OUTPUT, index=False, date_format="%Y-%m-%d")
print(f"Đã lưu {len(vnindex):,} dòng vào {VNINDEX_OUTPUT}")


## 6. Đọc và làm sạch file CSV vàng SJC tại TPHCM


In [ ]:
gold_input = next(
    (path.resolve() for path in GOLD_INPUT_CANDIDATES if path.exists()),
    None,
)
if gold_input is None:
    expected = " hoặc ".join(str(path) for path in GOLD_INPUT_CANDIDATES)
    raise FileNotFoundError(
        "Không tìm thấy file CSV giá vàng. Hãy đặt 3y-sjc.csv tại " + expected
    )

gold = pd.read_csv(gold_input)
required = ["query_date", "location", "product", "buy", "sell"]
missing = [column for column in required if column not in gold.columns]
if missing:
    raise ValueError(f"Dữ liệu vàng thiếu các cột bắt buộc: {missing}")

gold = gold.loc[
    gold["location"].eq("TPHCM") & gold["product"].eq("SJC")
].copy()
query_dates = gold["query_date"].astype(str).str.replace(".0", "", regex=False)
gold["Date"] = pd.to_datetime(query_dates, format="%Y%m%d", errors="coerce")
for column in ["buy", "sell"]:
    gold[column] = pd.to_numeric(gold[column], errors="coerce")
gold = gold.replace([np.inf, -np.inf], np.nan)
gold = gold.dropna(subset=["Date", "buy", "sell"])
gold = gold.loc[
    (gold["buy"] > 0)
    & (gold["sell"] > 0)
    & (gold["sell"] >= gold["buy"])
].copy()
gold = keep_requested_dates(gold)

sort_columns = ["Date"]
for column in ["updated_at", "crawl_time"]:
    if column in gold:
        gold[column] = pd.to_datetime(gold[column], errors="coerce")
        sort_columns.append(column)
gold = gold.sort_values(sort_columns, kind="stable")
gold = gold.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)
gold["gold_price"] = (gold["buy"] + gold["sell"]) / 2
gold = gold[["Date", "location", "product", "buy", "sell", "gold_price"]]
gold.to_csv(GOLD_OUTPUT, index=False, date_format="%Y-%m-%d")
print(f"Đã đọc {gold_input}")
print(f"Đã lưu {len(gold):,} dòng vào {GOLD_OUTPUT}")


## 7. Kiểm tra bốn file đầu ra


In [ ]:
outputs = {
    str(BTC_OUTPUT): (btc, "btc_price"),
    str(USDVND_OUTPUT): (usdvnd, "usdvnd_rate"),
    str(GOLD_OUTPUT): (gold, "gold_price"),
    str(VNINDEX_OUTPUT): (vnindex, "vnindex_price"),
}

summary = []
for filename, (data, price_column) in outputs.items():
    assert Path(filename).exists(), f"Chưa tạo được {filename}"
    assert not data.empty, f"{filename} không có dữ liệu"
    assert data["Date"].duplicated().sum() == 0, f"{filename} còn ngày trùng"
    assert data[price_column].notna().all(), f"{filename} còn giá thiếu"
    assert data[price_column].gt(0).all(), f"{filename} còn giá không dương"
    summary.append(
        {
            "file": filename,
            "rows": len(data),
            "start": data["Date"].min().strftime("%Y-%m-%d"),
            "end": data["Date"].max().strftime("%Y-%m-%d"),
        }
    )

pd.DataFrame(summary)
